# Textual Data Analysis Pipeline Dashboard

In [1]:
%load_ext autoreload
%autoreload 2

import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 匯入重構後的處理管線
from utils.pipeline_preprocess import run_preprocessing
from utils.pipeline_features import build_features
from utils.pipeline_modeling import evaluate_all_models

### Step 1: Data Preprocessing (資料前處理)
從 `data/raw_json` 讀取資料，萃取內容，並清除空白記錄。

**💡 執行這段會產生什麼？**
1. **實體檔案**：
   - `artifacts/reports/judgment_labels.xlsx`: 自動整理好的全案件勝敗訴、以及您剛設定的三種和解標籤。
   - `data/IP_Law_cases/`: 把過濾後的智財案件原始 JSON 單獨複製到這。
   - `artifacts/reports/fact_removed_blank.xlsx`: 取代原本龐大判決的乾淨「犯罪事實」與「理由」特徵文字檔。
2. **記憶體回傳值變數 (`df_clean`)**：內含判決 JID 與乾淨字串的 DataFrame，以及下方會印出預覽表格。

In [3]:
# 如果已經跑過，可將此行註解以節省時間
df_clean = run_preprocessing(
    input_folder="../data/raw_json", 
    output_folder="../data/IP_Law_cases", 
    artifacts_folder="../artifacts/reports", 
    n_jobs=-1  # n_jobs=1 為單純循序運算; n_jobs=-1 代表啟用平行運算使用所有資源

)
display(df_clean.head())

1. 執行 classify_cases 分類判決書 (純記憶體精簡版，無視 macOS I/O 限制)...


Classifying Cases: 100%|██████████| 90027/90027 [1:48:08<00:00, 13.87it/s]  


判決標籤已儲存至: ../artifacts/reports/judgment_labels.xlsx
2. 執行 extract_fact 萃取判決事實 (從原始 JSON 讀取，零 IPC 序列化開銷)...


Extracting Facts: 100%|██████████| 41570/41570 [00:41<00:00, 1000.66it/s]


3. 執行 remove_blank 移除空資料...
前處理完成！清理後的資料儲存於: ../artifacts/reports/fact_removed_blank.xlsx


,Text
JID,
"KLDM,101,智易,4,20130218,1",一、公訴意旨略以:被告周棟堅明知「VOLKSWAGEN」係德商福斯汽車股份有限公司下稱福斯公...
"KSDM,105,智簡,22,20161011,1","一、陳建鈞明知如附表一所示商標註冊審定號之商標名稱及圖樣,係附表一所示商標權人依法向我國經濟..."
"TPDM,93,簡,792,20040427,1","一、甲○○前於民國九十年間因竊盜案件,經臺灣板橋地方法院判處拘役四十日,於九十年八月二十二日..."
"TCDM,110,智簡,25,20210830,1","一、犯罪事實:洪金春明知如附表「商標註冊/審定號」所示之商標圖樣,分別係附表所示之商標權人向..."
"SLEM,98,士簡,1066,20091221,1","一、本件犯罪事實及證據併所犯法條均引用檢察官聲請簡易判決處刑書之記載,並補充:一犯罪事實欄一..."


### Step 2: Tokenization & DTM (斷詞與特徵矩陣)
使用 CKIP 建立斷詞，並計算 Bag-of-Words (BoW) 與 TF-IDF 矩陣。

**💡 執行這段會產生什麼？**
1. **實體檔案 (非常耗時，通常只需跑一次)**：
   - `artifacts/reports/word_seg.xlsx`: 儲存經過 CkipWordSegmenter 斷詞後的每個字串陣列。
   - `artifacts/reports/verdict_results.xlsx`: 對齊過斷詞順序的最終 One-Hot 標籤集。
   - `lexicon_resources/dtm_csr_BoW.npz`: Bag of Words 的 Sparse 特徵稀疏矩陣。
   - `lexicon_resources/dtm_csr_TF_IDF.npz`: TF-IDF 的 Sparse 特徵稀疏矩陣。
2. **記憶體回傳值變數 (`dtm_features`)**：回傳建立好的特徵矩陣以供查看維度。

In [ ]:
# 如果已經跑過並存至檔案，會自動讀取而跳過重複斷詞
dtm_features = build_features(
    df_clean_path="../artifacts/reports/fact_removed_blank.xlsx",
    artifacts_folder="../artifacts/reports",
    lexicon_folder="../lexicon_resources"
)

### Step 3: Modeling & Evaluation (模型訓練與評估)
呼叫 `algos/` 資料夾中的各種演算法進行驗證與測試。

**💡 執行這段會產生什麼？**
1. **終端/畫面輸出**：
   - 逐步顯示 Train/Test 的資料拆分大小。
   - MNIR 萃取特徵的轉換紀錄（`mnir_z_train_bow.npy`）。
   - 各個 Grid Search 超參數(如 SVM的 C 值)的尋優過程、最佳 F1-Mean 數值。
2. **實體檔案**：如果您後續擴充呼叫了 PyTorch 類神經網路，會另外存下最佳的 `.pt` 權重。
3. **記憶體回傳值變數 (`results_df`)**：回傳一個包含各模型 (SVM, NB, RF 等) 測試集 Accuracy 與 F1-Score 的 DataFrame，自動印出漂亮評估表供論文擷圖使用。

In [ ]:
results_df = evaluate_all_models(
    dtm_bow_path="../lexicon_resources/dtm_csr_BoW.npz",
    dtm_tfidf_path="../lexicon_resources/dtm_csr_TF_IDF.npz",
    verdict_results_path="../artifacts/reports/verdict_results.xlsx"
)

display(results_df)